# 03 — Carga da Camada Gold

Este notebook lê a Silver e produz datasets analíticos prontos para:
- **Dashboards** e relatórios executivos
- **Análises estatísticas** de desigualdade educacional
- **Treinamento de modelos de ML** (predição de alfabetização)

**Datasets produzidos**:
1. `tc02_indicador_por_municipio` — Dataset principal com contexto geográfico e social completo
2. `tc02_metas_vs_resultados` — Comparação entre metas e resultados reais por UF/ano
3. `tc02_evolucao_temporal` — Série histórica do indicador por nível geográfico
4. `tc02_ranking_municipios` — Ranking de municípios no ano mais recente

**Aplicação em IA**: a camada Gold alimenta modelos preditivos de alfabetização
e análises de cluster de vulnerabilidade educacional.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# Verifica pré-requisito
try:
    spark.read.table("silver.tc02_dim_uf").limit(1).count()
except Exception as err:
    raise ValueError(
        "Tabela 'silver.tc02_dim_uf' não encontrada. Execute 03_carga_camada_silver.py."
    ) from err

df_silver_uf    = spark.read.table("silver.tc02_dim_uf")
df_silver_municipio   = spark.read.table("silver.tc02_dim_municipio")
df_silver_meta_brasil = spark.read.table("silver.tc02_meta_brasil")
df_silver_meta_uf = spark.read.table("silver.tc02_meta_uf")
df_silver_meta_mun = spark.read.table("silver.tc02_meta_municipio")
df_silver_alunos = spark.read.table("silver.tc02_alunos")

print(f"Silver carregada: {df_silver_uf.count()} registros")

## Gold 0: Tratamento e criação da tabela fato alfabetização consolidada

In [0]:
# df_silver_alunos.display()
display(df_silver_municipio.limit(100))


In [0]:
# -*- coding: utf-8 -*-
"""
Tech Challenge - Fase 2
Pós Tech - Inteligência Artificial para Ciência de Dados
Projeto: Pipeline Híbrido para Análise da Alfabetização no Brasil

Script: pipeline_pyspark_gold_layer.py
Autor: Grupo de Engenharia de Dados (Gerado por NotebookLM)
Objetivo: Implementar a transformação dos dados tratados da Camada Silver
          para as Tabelas Fato e Dimensão na Camada Gold (Star Schema de Kimball),
          garantindo o alinhamento de escala (0 a 10) e práticas de FinOps.
"""

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, substring, round

def create_gold_layer():
    """
    Função principal que carrega os dados tratados da camada Silver,
    aplica a modelagem dimensional de Kimball (Star Schema) e cria as
    tabelas Fato e Dimensão prontas para a camada Gold.
    """
    # -------------------------------------------------------------------------
    # PASSO 0: Inicialização da Sessão Spark
    # -------------------------------------------------------------------------
    # Configuramos os parâmetros com base em boas práticas de performance.
    # Limitamos as partições de shuffle para o nosso volume de dados atual, otimizando o overhead de rede.
    spark = SparkSession.builder \
        .appName("GoldLayerTransformations") \
        .config("spark.sql.shuffle.partitions", "4") \
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic") \
        .getOrCreate()
        
    print("Sessão Spark inicializada com sucesso.")

    # -------------------------------------------------------------------------
    # PASSO 2: Criação da Tabela de Dimensão - dim_municipio (Geográfica)
    # -------------------------------------------------------------------------
    print("Iniciando a criação de 'dim_municipio'...")
    # Extraímos a lista única de ID de municípios para representar a dimensão geográfica.
    # Usamos o código de UF do IBGE (dois primeiros dígitos do ID do município) para mapear o estado.
    dim_municipio = df_silver_municipio \
        .select("id_municipio") \
        .distinct() \
        .withColumn("id_uf", substring(col("id_municipio"), 1, 2)) \
        .withColumn(
            "sigla_uf",
            when(col("id_uf") == "11", "RO")
            .when(col("id_uf") == "12", "AC")
            .when(col("id_uf") == "13", "AM")
            .when(col("id_uf") == "14", "RR")
            .when(col("id_uf") == "15", "PA")
            .when(col("id_uf") == "16", "AP")
            .when(col("id_uf") == "17", "TO")
            .when(col("id_uf") == "21", "MA")
            .when(col("id_uf") == "22", "PI")
            .when(col("id_uf") == "23", "CE")
            .when(col("id_uf") == "24", "RN")
            .when(col("id_uf") == "25", "PB")
            .when(col("id_uf") == "26", "PE")
            .when(col("id_uf") == "27", "AL")
            .when(col("id_uf") == "28", "SE")
            .when(col("id_uf") == "29", "BA")
            .when(col("id_uf") == "31", "MG")
            .when(col("id_uf") == "32", "ES")
            .when(col("id_uf") == "33", "RJ")
            .when(col("id_uf") == "35", "SP")
            .when(col("id_uf") == "41", "PR")
            .when(col("id_uf") == "42", "SC")
            .when(col("id_uf") == "43", "RS")
            .when(col("id_uf") == "50", "MS")
            .when(col("id_uf") == "51", "MT")
            .when(col("id_uf") == "52", "GO")
            .when(col("id_uf") == "53", "DF")
            .otherwise("Desconhecido")
        ) \
        .withColumn(
            "nome_estado",
            when(col("sigla_uf") == "RO", "Rondônia")
            .when(col("sigla_uf") == "AC", "Acre")
            .when(col("sigla_uf") == "AM", "Amazonas")
            .when(col("sigla_uf") == "RR", "Roraima")
            .when(col("sigla_uf") == "PA", "Pará")
            .when(col("sigla_uf") == "AP", "Amapá")
            .when(col("sigla_uf") == "TO", "Tocantins")
            .when(col("sigla_uf") == "MA", "Maranhão")
            .when(col("sigla_uf") == "PI", "Piauí")
            .when(col("sigla_uf") == "CE", "Ceará")
            .when(col("sigla_uf") == "RN", "Rio Grande do Norte")
            .when(col("sigla_uf") == "PB", "Paraíba")
            .when(col("sigla_uf") == "PE", "Pernambuco")
            .when(col("sigla_uf") == "AL", "Alagoas")
            .when(col("sigla_uf") == "SE", "Sergipe")
            .when(col("sigla_uf") == "BA", "Bahia")
            .when(col("sigla_uf") == "MG", "Minas Gerais")
            .when(col("sigla_uf") == "ES", "Espírito Santo")
            .when(col("sigla_uf") == "RJ", "Rio de Janeiro")
            .when(col("sigla_uf") == "SP", "São Paulo")
            .when(col("sigla_uf") == "PR", "Paraná")
            .when(col("sigla_uf") == "SC", "Santa Catarina")
            .when(col("sigla_uf") == "RS", "Rio Grande do Sul")
            .when(col("sigla_uf") == "MS", "Mato Grosso do Sul")
            .when(col("sigla_uf") == "MT", "Mato Grosso")
            .when(col("sigla_uf") == "GO", "Goiás")
            .when(col("sigla_uf") == "DF", "Distrito Federal")
            .otherwise("Desconhecido")
        ) \
        .select("id_municipio", "id_uf", "sigla_uf", "nome_estado")

    # -------------------------------------------------------------------------
    # PASSO 3: Criação de Tabela Fato - fato_desempenho (Nível Micro / Alunos)
    # -------------------------------------------------------------------------
    print("Iniciando a criação de 'fato_desempenho'...")
    # Tabela fato microfocada para o detalhe de alunos, perfeita para o treinamento de modelos de IA/ML
    # que buscam predizer vulnerabilidade de alfabetização ou evasão.
    fato_desempenho = df_silver_alunos \
        .withColumn("alfabetizado_int", col("alfabetizado").cast("integer")) \
        .withColumn("proficiencia_double", col("proficiencia").cast("double")) \
        .withColumn("peso_aluno_double", col("peso_aluno").cast("double")) \
        .withColumn("ano_int", col("ano").cast("integer")) \
        .select(
            col("id_aluno"),
            col("ano_int").alias("ano"),
            col("id_municipio"),
            col("id_escola"),
            col("caderno"),
            col("serie"),
            col("rede"),
            col("presenca"),
            col("preenchimento_caderno"),
            col("alfabetizado_int").alias("alfabetizado"),
            round(col("proficiencia_double"), 4).alias("proficiencia"),
            round(col("peso_aluno_double"), 4).alias("peso_aluno")
        )

    # -------------------------------------------------------------------------
    # PASSO 4: Criação de Tabela Fato - fato_alfabetizacao_consolidada (Nível Macro)
    # -------------------------------------------------------------------------
    print("Iniciando a criação de 'fato_alfabetizacao_consolidada'...")
    
    # 4.1: Agrupamos os microdados e calculamos o indicador real das turmas (proporção * 10 para escala 0-10)
    df_alunos_agregado = df_silver_alunos \
        .filter(col("presenca") == "1") \
        .groupBy("ano", "id_municipio", "rede") \
        .agg(
            F.count("id_aluno").alias("total_alunos_avaliados"),
            F.avg(col("proficiencia")).alias("media_proficiencia_alunos"),
            # Projeção matemática para escala 0 a 10 (Média de alfabetizados na base de 0.0 a 1.0 multiplicada por 10)
            (F.avg(col("alfabetizado").cast("double")) * 10.0).alias("taxa_alfabetizacao_real_escala_10")
        )

    # 4.2: Extraímos de forma pivotada a meta municipal do ano correspondente (escala 0 a 10)
    df_metas_projetadas = df_silver_meta_mun \
        .withColumn(
            "meta_alfabetizacao_projetada",
            when(col("ano") == 2024, col("meta_alfabetizacao_2024"))
            .when(col("ano") == 2025, col("meta_alfabetizacao_2025"))
            .when(col("ano") == 2026, col("meta_alfabetizacao_2026"))
            .when(col("ano") == 2027, col("meta_alfabetizacao_2027"))
            .when(col("ano") == 2028, col("meta_alfabetizacao_2028"))
            .when(col("ano") == 2029, col("meta_alfabetizacao_2029"))
            .when(col("ano") == 2030, col("meta_alfabetizacao_2030"))
            .otherwise(None) # Para anos anteriores a 2024, não há meta estabelecida na base
        ) \
        .select(
            col("ano").cast("integer").alias("ano"),
            col("id_municipio"),
            col("rede"),
            col("meta_alfabetizacao_projetada"),
            col("percentual_participacao").alias("meta_percentual_participacao"),
            col("nivel_alfabetizacao")
        )

    # 4.3: Realizamos as junções (joins) e consolidamos os indicadores com o desvio
    # O desvio agora é matematicamente correto, pois comparamos duas variáveis na mesma escala decimal [0.0 - 10.0]
    fato_alfabetizacao_consolidada = df_alunos_agregado \
        .join(df_metas_projetadas, ["ano", "id_municipio", "rede"], "inner") \
        .join(
            df_silver_municipio.select(
                col("ano").cast("integer").alias("ano"),
                "id_municipio",
                "rede",
                "taxa_alfabetizacao",
                "media_portugues"
            ),
            ["ano", "id_municipio", "rede"],
            "left"
        ) \
        .withColumn(
            "desvio_da_meta",
            col("taxa_alfabetizacao_real_escala_10") - col("meta_alfabetizacao_projetada")
        ) \
        .select(
            col("ano"),
            col("id_municipio"),
            col("rede"),
            col("total_alunos_avaliados"),
            round(col("media_proficiencia_alunos"), 2).alias("media_proficiencia_alunos"),
            round(col("taxa_alfabetizacao_real_escala_10"), 2).alias("taxa_alfabetizacao_real_escala_10"),
            round(col("meta_alfabetizacao_projetada"), 2).alias("meta_alfabetizacao_projetada"),
            round(col("desvio_da_meta"), 2).alias("desvio_da_meta"),
            round(col("meta_percentual_participacao"), 2).alias("meta_percentual_participacao"),
            col("nivel_alfabetizacao"),
            round(col("media_portugues"), 2).alias("media_portugues_municipio")
        )

    # -------------------------------------------------------------------------
    # PASSO 5: Persistência Física dos Dados (FinOps e Performance)
    # -------------------------------------------------------------------------
    print("Iniciando a persistência física na camada Gold...")
    
    # Gravamos as tabelas no formato colunar Parquet (ou Delta Lake) para otimizar I/O.
    # Particionamos as tabelas fato por 'ano' para habilitar Partition Pruning em queries analíticas (FinOps).
    print(f"Salvando 'dim_municipio'...")
    dim_municipio.write \
        .mode("overwrite") \
        .saveAsTable("workspace.gold.tc02_dim_municipio")
        
    print(f"Salvando 'fato_desempenho' particionada por ano...")
    fato_desempenho.write \
        .mode("overwrite") \
        .partitionBy("ano") \
        .saveAsTable("workspace.gold.tc02_fato_desempenho")
        
    print(f"Salvando 'fato_alfabetizacao_consolidada' particionada por ano...")
    fato_alfabetizacao_consolidada.write \
        .mode("overwrite") \
        .partitionBy("ano") \
        .saveAsTable("workspace.gold.tc02_fato_alfabetizacao_consolidada")
        
    print("Gold Layer atualizada com sucesso de acordo com Kimball e FinOps!")
    spark.stop()

# if __name__ == "__main__":
#     # Define caminhos padrão executáveis para o pipeline local ou em cluster
create_gold_layer()


In [0]:
df_gold_dim_municipio = spark.read.table("gold.tc02_dim_municipio")
df_gold_fato_desempenho = spark.read.table("gold.tc02_fato_desempenho")
df_gold_fato_alfabetizacao_consolidada = spark.read.table("gold.tc02_fato_alfabetizacao_consolidada")

print("Base gold carregada com sucesso")

display(df_gold_dim_municipio.limit(100))
display(df_gold_fato_alfabetizacao_consolidada.limit(100))
display(df_gold_fato_desempenho.limit(100))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when

# -------------------------------------------------------------------------
# PASSO 1: Agregação da base de alunos (Geração de taxas reais na escala de 0 a 10)
# -------------------------------------------------------------------------
# Para fazer a correta Feature Engineering na camada Gold:
# 1. Filtramos apenas alunos presentes [1].
# 2. Convertemos a flag de 'alfabetizado' (0 ou 1) para double.
# 3. Multiplicamos a média por 10.0 para mapear o indicador para a escala [0.0 - 10.0].
df_alunos_agregado = df_bz_alunos \
    .filter(col("presenca") == "1") \
    .groupBy("ano", "id_municipio", "rede") \
    .agg(
        F.count("id_aluno").alias("total_alunos_avaliados"),
        F.avg(col("proficiencia")).alias("media_proficiencia_alunos"), # Média Saeb (ponto de corte 743) [2]
        # Projeção matemática direta para a escala 0 a 10
        (F.avg(col("alfabetizado").cast("double")) * 10.0).alias("taxa_alfabetizacao_real_escala_10")
    )

# -------------------------------------------------------------------------
# PASSO 2: Extração Dinâmica da Meta Municipal (Escala de 0 a 10)
# -------------------------------------------------------------------------
# Buscamos a meta correspondente ao ano corrente do registro para criar uma dimensão temporal.
df_metas_tratadas = df_bz_meta_mun \
    .withColumn(
        "meta_alfabetizacao_projetada",
        when(col("ano") == 2024, col("meta_alfabetizacao_2024"))
        .when(col("ano") == 2025, col("meta_alfabetizacao_2025"))
        .when(col("ano") == 2026, col("meta_alfabetizacao_2026"))
        .when(col("ano") == 2027, col("meta_alfabetizacao_2027"))
        .when(col("ano") == 2028, col("meta_alfabetizacao_2028"))
        .when(col("ano") == 2029, col("meta_alfabetizacao_2029"))
        .when(col("ano") == 2030, col("meta_alfabetizacao_2030")) # Meta final até 2030 [3]
        .otherwise(None)
    ) \
    .select(
        "ano", 
        "id_municipio", 
        "rede", 
        "meta_alfabetizacao_projetada", # Já é float de 0 a 10
        "percentual_participacao"
    )

# -------------------------------------------------------------------------
# PASSO 3: Junção (Joins) e Construção da Fato Consolidada na Camada Gold
# -------------------------------------------------------------------------
# Cruzamos os indicadores calculados, a tabela de metas e os dados gerais de município.
# As junções são feitas no Spark SQL para mitigar o gargalo de consultas lentas no BI [4, 5].
fato_alfabetizacao_consolidada = df_alunos_agregado \
    .join(df_metas_tratadas, ["ano", "id_municipio", "rede"], "inner") \
    .join(
        df_bz_municipio.select(
            "ano", "id_municipio", "rede", "taxa_alfabetizacao", "media_portugues",
            "proporcao_aluno_nivel_0", "proporcao_aluno_nivel_1", "proporcao_aluno_nivel_2"
        ), 
        ["ano", "id_municipio", "rede"], 
        "inner"
    ) \
    .withColumn(
        # Agora o desvio é matematicamente correto (escala 0-10 vs escala 0-10)
        "desvio_da_meta", 
        col("taxa_alfabetizacao_real_escala_10") - col("meta_alfabetizacao_projetada")
    )

# -------------------------------------------------------------------------
# PASSO 4: Salvamento Otimizado (FinOps)
# -------------------------------------------------------------------------
# Persistimos a tabela Fato em Delta Lake, particionada por "ano" para Partition Pruning [6].
fato_alfabetizacao_consolidada.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("ano") \
    .save("/mnt/gold/fato_alfabetizacao_consolidada")

## Gold 1: Indicador por Município (Dataset Principal)

Dataset completo com contexto geográfico, social e comparativo com metas.
Inclui classificação por faixa de desempenho e gap em relação à meta.

In [0]:
df_gold_indicador_municipio = (
    df_silver
    .select(
        "id_municipio",
        "nome_municipio",
        "sigla_uf",
        "nome_uf",
        "regiao",
        "capital",
        "populacao_estimada",
        "ano",
        "total_alunos_2o_ano",
        "alunos_alfabetizados",
        "indicador_crianca_alfabetizada",
        "meta_nacional",
        "meta_uf",
        "ponto_corte_saeb",
    )
    .withColumn(
        "meta_referencia",
        F.coalesce(F.col("meta_uf"), F.col("meta_nacional"))
    )
    .withColumn(
        "gap_meta",
        F.round(F.col("meta_referencia") - F.col("indicador_crianca_alfabetizada"), 2)
    )
    .withColumn(
        "atingiu_meta",
        F.col("indicador_crianca_alfabetizada") >= F.col("meta_referencia")
    )
    .withColumn(
        "faixa_desempenho",
        F.when(F.col("indicador_crianca_alfabetizada") >= 70, "Alta (≥70%)")
         .when(F.col("indicador_crianca_alfabetizada") >= 55, "Média (55–70%)")
         .when(F.col("indicador_crianca_alfabetizada") >= 40, "Baixa (40–55%)")
         .otherwise("Crítica (<40%)")
    )
    .withColumn(
        "categoria_capital",
        F.when(F.col("capital"), "Capital").otherwise("Interior")
    )
    .withColumn("_data_processamento_gold", F.current_timestamp())
)

print(f"Gold Indicador Município: {df_gold_indicador_municipio.count()} registros")
display(
    df_gold_indicador_municipio
    .orderBy("sigla_uf", "ano")
    .select("nome_municipio", "sigla_uf", "regiao", "ano",
            "indicador_crianca_alfabetizada", "meta_referencia", "gap_meta", "faixa_desempenho")
    .limit(20)
)

## Gold 2: Metas vs Resultados por UF

Agrega os resultados por UF e ano, calcula o gap em relação às metas
e classifica a situação de cada estado.

In [0]:
df_gold_metas_resultados = (
    df_silver
    .groupBy("sigla_uf", "nome_uf", "regiao", "ano")
    .agg(
        F.round(F.avg("indicador_crianca_alfabetizada"), 2).alias("indicador_medio_uf"),
        F.round(F.min("indicador_crianca_alfabetizada"), 2).alias("indicador_min_uf"),
        F.round(F.max("indicador_crianca_alfabetizada"), 2).alias("indicador_max_uf"),
        F.round(F.stddev("indicador_crianca_alfabetizada"), 2).alias("desvio_padrao_uf"),
        F.first("meta_uf").alias("meta_uf"),
        F.first("meta_nacional").alias("meta_nacional"),
        F.sum("total_alunos_2o_ano").alias("total_alunos_uf"),
        F.sum("alunos_alfabetizados").alias("total_alfabetizados_uf"),
        F.countDistinct("id_municipio").alias("qtd_municipios"),
    )
    .withColumn("meta_referencia", F.coalesce(F.col("meta_uf"), F.col("meta_nacional")))
    .withColumn(
        "gap_meta",
        F.round(F.col("meta_referencia") - F.col("indicador_medio_uf"), 2)
    )
    .withColumn("atingiu_meta", F.col("indicador_medio_uf") >= F.col("meta_referencia"))
    .withColumn(
        "situacao",
        F.when(F.col("indicador_medio_uf") >= F.col("meta_referencia"), "Meta Atingida")
         .when(F.col("gap_meta") <= 5.0, "Próximo da Meta (≤5pp)")
         .otherwise("Abaixo da Meta")
    )
    .withColumn("_data_processamento_gold", F.current_timestamp())
    .orderBy("ano", "gap_meta")
)

print(f"Gold Metas vs Resultados: {df_gold_metas_resultados.count()} registros")
display(df_gold_metas_resultados.select(
    "sigla_uf", "nome_uf", "regiao", "ano", "indicador_medio_uf",
    "meta_referencia", "gap_meta", "situacao", "qtd_municipios"
).limit(30))

## Gold 3: Evolução Temporal

Série histórica do indicador em dois níveis: **Brasil** e **por UF**.
Inclui variação ano a ano (pontos percentuais) para análise de tendência.

In [0]:
janela_evolucao = Window.partitionBy("nivel", "referencia").orderBy("ano")

df_evolucao_brasil = (
    df_silver
    .groupBy("ano")
    .agg(
        F.round(F.avg("indicador_crianca_alfabetizada"), 2).alias("indicador_medio"),
        F.sum("total_alunos_2o_ano").alias("total_alunos"),
        F.sum("alunos_alfabetizados").alias("total_alfabetizados"),
        F.first("meta_nacional").alias("meta"),
    )
    .withColumn("nivel", F.lit("BRASIL"))
    .withColumn("referencia", F.lit("BRASIL"))
    .withColumn("nome_referencia", F.lit("Brasil"))
    .withColumn("regiao", F.lit("Nacional"))
)

df_evolucao_uf = (
    df_silver
    .groupBy("sigla_uf", "nome_uf", "regiao", "ano")
    .agg(
        F.round(F.avg("indicador_crianca_alfabetizada"), 2).alias("indicador_medio"),
        F.sum("total_alunos_2o_ano").alias("total_alunos"),
        F.sum("alunos_alfabetizados").alias("total_alfabetizados"),
        F.first("meta_uf").alias("meta"),
    )
    .withColumn("nivel", F.lit("UF"))
    .withColumnRenamed("sigla_uf", "referencia")
    .withColumnRenamed("nome_uf", "nome_referencia")
)

colunas_comuns = ["ano", "nivel", "referencia", "nome_referencia", "regiao",
                  "indicador_medio", "total_alunos", "total_alfabetizados", "meta"]

df_gold_evolucao = (
    df_evolucao_brasil.select(*colunas_comuns)
    .unionByName(df_evolucao_uf.select(*colunas_comuns))
    .withColumn(
        "variacao_pp",
        F.round(
            F.col("indicador_medio") - F.lag("indicador_medio", 1).over(janela_evolucao), 2
        )
    )
    .withColumn(
        "tendencia",
        F.when(F.col("variacao_pp") > 0, "Melhora")
         .when(F.col("variacao_pp") < 0, "Piora")
         .otherwise("Estável")
    )
    .withColumn("_data_processamento_gold", F.current_timestamp())
)

print(f"Gold Evolução Temporal: {df_gold_evolucao.count()} registros")
display(
    df_gold_evolucao
    .filter(F.col("nivel") == "BRASIL")
    .orderBy("ano")
    .select("ano", "nivel", "referencia", "indicador_medio", "meta", "variacao_pp", "tendencia")
)

## Gold 4: Ranking de Municípios

Ranking nacional e regional de municípios no ano mais recente.
Útil para identificar boas práticas (top performers) e municípios vulneráveis.

In [0]:
ano_mais_recente = df_silver.agg(F.max("ano")).first()[0]

janela_nacional = Window.orderBy(F.col("indicador_crianca_alfabetizada").desc())
janela_regiao   = Window.partitionBy("regiao").orderBy(F.col("indicador_crianca_alfabetizada").desc())
janela_uf       = Window.partitionBy("sigla_uf").orderBy(F.col("indicador_crianca_alfabetizada").desc())

df_gold_ranking = (
    df_silver
    .filter(F.col("ano") == ano_mais_recente)
    .select(
        "id_municipio", "nome_municipio", "sigla_uf", "nome_uf", "regiao",
        "capital", "populacao_estimada",
        "indicador_crianca_alfabetizada",
        "total_alunos_2o_ano", "alunos_alfabetizados",
        "meta_nacional", "meta_uf",
    )
    .withColumn("ranking_nacional", F.rank().over(janela_nacional))
    .withColumn("ranking_regiao",   F.rank().over(janela_regiao))
    .withColumn("ranking_uf",       F.rank().over(janela_uf))
    .withColumn(
        "quartil_nacional",
        F.when(F.col("ranking_nacional") <= F.count("*").over(Window.orderBy(F.lit(1))) * 0.25, "Q1 - Top 25%")
         .when(F.col("ranking_nacional") <= F.count("*").over(Window.orderBy(F.lit(1))) * 0.50, "Q2 - 25-50%")
         .when(F.col("ranking_nacional") <= F.count("*").over(Window.orderBy(F.lit(1))) * 0.75, "Q3 - 50-75%")
         .otherwise("Q4 - Bottom 25%")
    )
    .withColumn("_data_processamento_gold", F.current_timestamp())
)

print(f"Gold Ranking Municípios (ano {ano_mais_recente}): {df_gold_ranking.count()} registros")
display(
    df_gold_ranking
    .orderBy("ranking_nacional")
    .select("ranking_nacional", "nome_municipio", "sigla_uf", "regiao",
            "indicador_crianca_alfabetizada", "capital", "quartil_nacional")
    .limit(20)
)

## Escrita na Camada Gold

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

tabelas_gold = {
    "gold.tc02_indicador_por_municipio": df_gold_indicador_municipio,
    "gold.tc02_metas_vs_resultados":     df_gold_metas_resultados,
    "gold.tc02_evolucao_temporal":       df_gold_evolucao,
    "gold.tc02_ranking_municipios":      df_gold_ranking,
}

for nome, df in tabelas_gold.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome)
    )

print("Tabelas Gold criadas com sucesso:")
for nome in tabelas_gold:
    print(f"  - {nome} => {spark.read.table(nome).count()} linhas")

## Resumo da Camada Gold

In [0]:
print("=" * 60)
print("SUMÁRIO EXECUTIVO — PIPELINE BATCH CONCLUÍDO")
print("=" * 60)

total_municipios = spark.read.table("gold.tc02_indicador_por_municipio").select("id_municipio").distinct().count()
anos_disponiveis = sorted([r.ano for r in spark.read.table("gold.tc02_evolucao_temporal").filter(F.col("nivel") == "BRASIL").select("ano").collect()])
indicador_br = spark.read.table("gold.tc02_evolucao_temporal").filter(
    (F.col("nivel") == "BRASIL") & (F.col("ano") == max(anos_disponiveis))
).first()

print(f"\nMunicípios cobertos:          {total_municipios}")
print(f"Período analisado:            {min(anos_disponiveis)}–{max(anos_disponiveis)}")
print(f"Indicador Brasil ({max(anos_disponiveis)}):   {indicador_br['indicador_medio']}%")

acima_meta = spark.read.table("gold.tc02_metas_vs_resultados").filter(
    (F.col("ano") == max(anos_disponiveis)) & F.col("atingiu_meta")
).count()
total_ufs = spark.read.table("gold.tc02_metas_vs_resultados").filter(
    F.col("ano") == max(anos_disponiveis)
).count()
print(f"UFs que atingiram a meta:     {acima_meta}/{total_ufs}")
print(f"\nTabelas Gold disponíveis:")
for nome in tabelas_gold:
    print(f"  - {nome}")

print("\nPróximo passo: executar 05_streaming_simulacao.py")